# Tutorial: Computing Pathway Activity Scores from Tahoe-100M

This notebook demonstrates how to:
1. Load pseudobulk differential expression data from Tahoe-100M
2. Load all relevant metadata (genes, drugs, cell lines)
3. Compute pathway activity scores by aggregating gene-level differential expression

The pseudobulk DE data is already averaged across cells per treatment condition, making it ideal for pathway-level analysis without downloading the full 500GB single-cell dataset.

## Import Libraries

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from pathlib import Path

## Load All Metadata Tables

First, let's load the metadata tables which are small enough to fit in memory.

In [2]:
# gene metadata: maps token IDs to gene symbols and Ensembl IDs
gene_metadata = load_dataset("tahoebio/Tahoe-100M", name="gene_metadata", split="train").to_pandas()
print(f"Gene metadata: {len(gene_metadata)} genes")
gene_metadata.head()

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Gene metadata: 62710 genes


,gene_symbol,ensembl_id,token_id
0,TSPAN6,ENSG00000000003,3
1,TNMD,ENSG00000000005,4
2,DPM1,ENSG00000000419,5
3,SCYL3,ENSG00000000457,6
4,C1orf112,ENSG00000000460,7


In [3]:
# drug metadata: MOA, targets, SMILES, etc.
drug_metadata = load_dataset("tahoebio/Tahoe-100M", name="drug_metadata", split="train").to_pandas()
print(f"Drug metadata: {len(drug_metadata)} drugs")
drug_metadata.head()

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Drug metadata: 379 drugs


,drug,targets,moa-broad,moa-fine,human-approved,clinical-trials,gpt-notes-approval,canonical_smiles,pubchem_cid
0,Talc,None,unclear,unclear,yes,yes,Talc used in pharma and cosmetics; safety unde...,[OH-].[OH-].[O-][Si]12O[Si]3(O[Si](O1)(O[Si](O...,165411828.0
1,Bortezomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma and mantle cell ...,B(C(CC(C)C)NC(=O)C(CC1=CC=CC=C1)NC(=O)C2=NC=CN...,387447.0
2,Ixazomib,PSMB5,inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment.,B(C(CC(C)C)NC(=O)CNC(=O)C1=C(C=CC(=C1)Cl)Cl)(O)O,25183872.0
3,Ixazomib citrate,"PSMB1, PSMB2, PSMB5",inhibitor/antagonist,Proteasome inhibitor,yes,yes,Approved for multiple myeloma treatment as par...,B1(OC(=O)C(O1)(CC(=O)O)CC(=O)O)C(CC(C)C)NC(=O)...,56844015.0
4,Lactate (calcium),None,unclear,unclear,yes,yes,"Used in medical settings, but not specifically...",C.CC(C(=O)[O-])O.[Ca+2],168311648.0


In [4]:
# cell line metadata: driver mutations, organ type, etc.
cell_line_metadata = load_dataset("tahoebio/Tahoe-100M", name="cell_line_metadata", split="train").to_pandas()
print(f"Cell line metadata: {len(cell_line_metadata)} entries ({cell_line_metadata['cell_name'].nunique()} unique cell lines)")
cell_line_metadata.head()

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Cell line metadata: 1000 entries (102 unique cell lines)


,cell_name,Cell_ID_DepMap,Cell_ID_Cellosaur,Organ,Driver_Gene_Symbol,Driver_VarZyg,Driver_VarType,Driver_ProtEffect_or_CdnaEffect,Driver_Mech_InferDM,Driver_GeneType_DM
0,A549,ACH-000681,CVCL_0023,Lung,CDKN2A,Hom,Deletion,DEL,LoF,Suppressor
1,A549,ACH-000681,CVCL_0023,Lung,CDKN2B,Hom,Deletion,DEL,LoF,Suppressor
2,A549,ACH-000681,CVCL_0023,Lung,KRAS,Hom,Missense,p.G12S,GoF,Oncogene
3,A549,ACH-000681,CVCL_0023,Lung,SMARCA4,Hom,Frameshift,p.Q729fs,LoF,Suppressor
4,A549,ACH-000681,CVCL_0023,Lung,STK11,Hom,Stopgain,p.Q37*,LoF,Suppressor


In [5]:
# sample metadata: drug concentrations and QC metrics
sample_metadata = load_dataset("tahoebio/Tahoe-100M", name="sample_metadata", split="train").to_pandas()
print(f"Sample metadata: {len(sample_metadata)} samples")
sample_metadata.head()

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Sample metadata: 1344 samples


,sample,plate,mean_gene_count,mean_tscp_count,mean_mread_count,mean_pcnt_mito,drug,drugname_drugconc
0,smp_1495,plate1,1354.169768,2027.115940,2444.032416,0.033956,Infigratinib,"[('Infigratinib', 0.05, 'uM')]"
1,smp_1496,plate1,1404.454157,2226.282791,2690.685970,0.071723,Erdafitinib,"[('Erdafitinib ', 0.05, 'uM')]"
2,smp_1497,plate1,1205.267794,1859.375821,2246.200127,0.084853,Everolimus,"[('Everolimus', 0.05, 'uM')]"
3,smp_1498,plate1,1225.510822,1906.494566,2298.907623,0.088262,Pemigatinib,"[('Pemigatinib', 0.05, 'uM')]"
4,smp_1499,plate1,1231.372881,1861.305085,2245.372881,0.050802,Abemaciclib,"[('Abemaciclib', 0.05, 'uM')]"


## Load Pseudobulk Differential Expression Data

The `pseudobulk_differential_expression` configuration contains pre-computed differential expression statistics per gene, per treatment condition. This is already aggregated across cells.

**Columns include:**
- `gene_name`: Gene symbol
- `log2FoldChange`: Effect size (treatment vs control)
- `padj`: Adjusted p-value
- `drug`, `cell_line_id`, `concentration`: Treatment identifiers
- `n_cells_trt`, `n_cells_ctrl`: Number of cells in each condition

We'll use streaming to avoid downloading the full ~50GB dataset.

In [6]:
# load with streaming to avoid downloading full dataset
de_dataset = load_dataset(
    "tahoebio/Tahoe-100M", 
    name="pseudobulk_differential_expression", 
    split="train",
    streaming=True
)

print("Dataset loaded in streaming mode")
print(f"Features: {de_dataset.features}")

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1026 [00:00<?, ?it/s]

Dataset loaded in streaming mode
Features: {'gene_name': Value('string'), 'baseMean': Value('float32'), 'log2FoldChange': Value('float32'), 'lfcSE': Value('float32'), 'stat': Value('float32'), 'pvalue': Value('float32'), 'padj': Value('float32'), 'plate': Value('string'), 'n_cells_trt': Value('int64'), 'n_cells_ctrl': Value('int64'), 'Cell_ID_Cellosaur': Value('string'), 'Cell_ID_DepMap': Value('string'), 'drug': Value('string'), 'concentration': Value('float32'), 'concentration_unit': Value('string'), 'Cell_Name_Vevo': Value('string')}


In [7]:
# sample 10 rows to inspect the structure
sample_rows = list(de_dataset.take(10))
de_sample = pd.DataFrame(sample_rows)
print(f"Sample of {len(de_sample)} rows:")
de_sample

Sample of 10 rows:


,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,plate,n_cells_trt,n_cells_ctrl,Cell_ID_Cellosaur,Cell_ID_DepMap,drug,concentration,concentration_unit,Cell_Name_Vevo
0,TSPAN6,16.415594,-0.279559,0.366763,-0.762235,0.445920,0.756971,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
1,TNMD,0.000000,NaN,NaN,NaN,NaN,NaN,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
2,DPM1,130.059631,-0.050576,0.137905,-0.366746,0.713808,0.897968,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
3,SCYL3,14.038391,-0.794847,0.446285,-1.781028,0.074908,0.348144,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
4,C1orf112,27.576792,-0.020508,0.335832,-0.061065,0.951307,0.983305,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
5,FGR,0.952125,3.055907,1.857289,1.645359,0.099896,NaN,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
6,CFH,11.456981,-0.595210,0.454868,-1.308533,0.190693,0.537274,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
7,FUCA2,33.008358,0.106479,0.279080,0.381535,0.702806,0.892402,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
8,GCLC,219.124878,-0.196621,0.107233,-1.833595,0.066714,0.327625,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
9,NFYA,49.612526,0.034212,0.248056,0.137921,0.890303,0.964077,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549


In [9]:
# inspect column types and values
print("\nColumn info:")
for col in de_sample.columns:
    print(f"  {col}: {de_sample[col].dtype} - example: {de_sample[col].iloc[0]}")


Column info:
  gene_name: object - example: TSPAN6
  baseMean: float64 - example: 16.41559410095215
  log2FoldChange: float64 - example: -0.27955949306488037
  lfcSE: float64 - example: 0.3667627274990082
  stat: float64 - example: -0.7622353434562683
  pvalue: float64 - example: 0.44591957330703735
  padj: float64 - example: 0.7569714784622192
  plate: object - example: 1
  n_cells_trt: int64 - example: 1378
  n_cells_ctrl: int64 - example: 4862
  Cell_ID_Cellosaur: object - example: CVCL_0023
  Cell_ID_DepMap: object - example: ACH-000681
  drug: object - example: 4EGI-1
  concentration: float64 - example: 0.05000000074505806
  concentration_unit: object - example: uM
  Cell_Name_Vevo: object - example: A549


## Define Pathway Gene Signatures

We'll use pre-defined gene signatures for key biological pathways. These can be extended with MSigDB or other pathway databases.

In [8]:
# load existing gene signatures if available, otherwise define them
signatures_path = Path("../data/processed/gene_signatures.json")

if signatures_path.exists():
    with open(signatures_path) as f:
        PATHWAY_SIGNATURES = json.load(f)
    print(f"Loaded {len(PATHWAY_SIGNATURES)} pathway signatures from file")
else:
    # define pathway signatures manually
    PATHWAY_SIGNATURES = {
        "proliferation": [
            "MKI67", "PCNA", "TOP2A", "MCM2", "MCM3", "MCM4", "MCM5", "MCM6", "MCM7",
            "CDK1", "CCNB1", "CCNB2", "CCNA2", "PLK1", "AURKA", "AURKB", "BUB1",
            "BUB1B", "CDC20", "FOXM1", "E2F1", "TYMS"
        ],
        "apoptosis": [
            "BCL2", "BAX", "BAK1", "BID", "PUMA", "NOXA", "CASP3", "CASP7",
            "CASP8", "CASP9", "CYCS", "APAF1", "PARP1", "XIAP", "BIRC5"
        ],
        "mapk_pathway": [
            "EGFR", "KRAS", "NRAS", "HRAS", "BRAF", "RAF1", "ARAF", "MAP2K1",
            "MAP2K2", "MAPK1", "MAPK3", "DUSP1", "DUSP4", "DUSP6", "ETV1",
            "ETV4", "ETV5", "SPRY1", "SPRY2", "SPRY4"
        ]
    }
    print(f"Using {len(PATHWAY_SIGNATURES)} built-in pathway signatures")

# show signature sizes
for name, genes in PATHWAY_SIGNATURES.items():
    print(f"  {name}: {len(genes)} genes")

# create a flat set of all signature genes for filtering
all_signature_genes = set()
for genes in PATHWAY_SIGNATURES.values():
    all_signature_genes.update(genes)
print(f"\nTotal unique genes across all signatures: {len(all_signature_genes)}")

Loaded 3 pathway signatures from file
  proliferation: 22 genes
  apoptosis: 15 genes
  mapk_pathway: 20 genes

Total unique genes across all signatures: 57


## Load Differential Expression for Signature Genes

Now we'll stream through the DE data and collect only the rows for our signature genes. This avoids loading the full dataset.

In [9]:
def collect_signature_de(dataset, signature_genes, max_rows=100000):
    """
    Stream through DE data and collect rows matching signature genes.
    
    Args:
        dataset: HuggingFace streaming dataset
        signature_genes: set of gene symbols to keep
        max_rows: maximum total rows to scan (for demo purposes)
    
    Returns:
        DataFrame with DE data for signature genes
    """
    collected = []
    scanned = 0
    
    for row in dataset:
        scanned += 1
        if row['gene_name'] in signature_genes:
            collected.append(row)
        
        if scanned >= max_rows:
            break
        
        if scanned % 10000 == 0:
            print(f"  Scanned {scanned:,} rows, collected {len(collected)} signature genes...")
    
    print(f"\nDone! Scanned {scanned:,} rows, collected {len(collected)} for signature genes")
    return pd.DataFrame(collected)

In [10]:
# reload the streaming dataset (iterators are consumed)
de_dataset = load_dataset(
    "tahoebio/Tahoe-100M", 
    name="pseudobulk_differential_expression", 
    split="train",
    streaming=True
)

# collect DE data for signature genes
# note: for the full dataset, you'd want to use dask or increase max_rows significantly
de_signatures = collect_signature_de(de_dataset, all_signature_genes, max_rows=100000)
de_signatures.head()

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1026 [00:00<?, ?it/s]

  Scanned 10,000 rows, collected 35 signature genes...
  Scanned 20,000 rows, collected 53 signature genes...
  Scanned 30,000 rows, collected 54 signature genes...
  Scanned 40,000 rows, collected 55 signature genes...
  Scanned 50,000 rows, collected 55 signature genes...
  Scanned 60,000 rows, collected 55 signature genes...
  Scanned 70,000 rows, collected 85 signature genes...
  Scanned 80,000 rows, collected 108 signature genes...
  Scanned 90,000 rows, collected 109 signature genes...

Done! Scanned 100,000 rows, collected 110 for signature genes


,gene_name,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,plate,n_cells_trt,n_cells_ctrl,Cell_ID_Cellosaur,Cell_ID_DepMap,drug,concentration,concentration_unit,Cell_Name_Vevo
0,ETV1,7.918274,0.024483,0.509920,0.048013,0.961706,0.987555,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
1,BID,50.379326,-0.174893,0.369805,-0.472933,0.636261,0.864527,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
2,BAK1,12.787655,-0.182853,0.407235,-0.449009,0.653425,0.872716,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
3,CASP8,77.580399,-0.106274,0.181169,-0.586599,0.557473,0.822790,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549
4,MCM2,10.017890,0.072306,0.469447,0.154024,0.877591,0.958634,1,1378,4862,CVCL_0023,ACH-000681,4EGI-1,0.05,uM,A549


## Compute Pathway Activity Scores

For each treatment condition (drug + cell line + concentration), we compute a pathway activity score as the mean log2FoldChange across genes in that pathway.

In [11]:
def compute_pathway_scores(de_df, pathway_signatures):
    """
    Compute pathway activity scores from gene-level differential expression.
    
    For each pathway, computes:
    - mean log2FoldChange across member genes
    - min adjusted p-value (most significant)
    - number of genes detected
    
    Args:
        de_df: DataFrame with columns [gene_name, log2FoldChange, padj, drug, cell_line_id, concentration]
        pathway_signatures: dict mapping pathway names to gene lists
    
    Returns:
        DataFrame with pathway scores per treatment condition
    """
    results = []
    
    # identify treatment condition columns
    # these may vary depending on the exact schema
    group_cols = ['drug']
    if 'cell_line_id' in de_df.columns:
        group_cols.append('cell_line_id')
    if 'concentration' in de_df.columns:
        group_cols.append('concentration')
    
    for pathway_name, pathway_genes in pathway_signatures.items():
        # filter to genes in this pathway
        pathway_df = de_df[de_df['gene_name'].isin(pathway_genes)].copy()
        
        if len(pathway_df) == 0:
            print(f"  Warning: No genes found for pathway '{pathway_name}'")
            continue
        
        # aggregate per treatment condition
        agg_dict = {
            'log2FoldChange': 'mean',
            'gene_name': 'count'
        }
        if 'padj' in pathway_df.columns:
            agg_dict['padj'] = 'min'
        
        scores = pathway_df.groupby(group_cols).agg(agg_dict).reset_index()
        
        # rename columns
        scores = scores.rename(columns={
            'log2FoldChange': f'{pathway_name}_log2fc',
            'padj': f'{pathway_name}_padj',
            'gene_name': f'{pathway_name}_n_genes'
        })
        
        # add percent change column
        # log2FC of -1 = 50% reduction, log2FC of +1 = 100% increase
        scores[f'{pathway_name}_pct_change'] = (2 ** scores[f'{pathway_name}_log2fc'] - 1) * 100
        
        scores['pathway'] = pathway_name
        results.append(scores)
    
    if not results:
        return pd.DataFrame()
    
    return results

In [12]:
# compute pathway scores
if len(de_signatures) > 0:
    pathway_scores_list = compute_pathway_scores(de_signatures, PATHWAY_SIGNATURES)
    
    for scores in pathway_scores_list:
        pathway_name = scores['pathway'].iloc[0]
        print(f"\n{pathway_name.upper()} pathway scores:")
        print(scores.head(10))
else:
    print("No signature genes found in the sampled data. Try increasing max_rows.")


PROLIFERATION pathway scores:
       drug  concentration  proliferation_log2fc  proliferation_n_genes  \
0    4EGI-1           0.05              0.039984                     22   
1  9-ING-41           0.05              0.157286                     22   

   proliferation_padj  proliferation_pct_change        pathway  
0            0.128359                  2.810245  proliferation  
1            0.013264                 11.518719  proliferation  

APOPTOSIS pathway scores:
       drug  concentration  apoptosis_log2fc  apoptosis_n_genes  \
0    4EGI-1           0.05         -0.154550                 13   
1  9-ING-41           0.05         -0.051164                 13   

   apoptosis_padj  apoptosis_pct_change    pathway  
0        0.243384            -10.158727  apoptosis  
1        0.210264             -3.484256  apoptosis  

MAPK_PATHWAY pathway scores:
       drug  concentration  mapk_pathway_log2fc  mapk_pathway_n_genes  \
0    4EGI-1           0.05            -0.072396          

## Merge with Metadata

Enrich the pathway scores with drug and cell line annotations.

In [13]:
def enrich_with_metadata(scores_df, drug_metadata, cell_line_metadata):
    """
    Merge pathway scores with drug and cell line metadata.
    """
    # merge drug info
    if 'drug' in scores_df.columns:
        drug_cols = ['drug', 'moa-fine', 'moa-broad', 'targets']
        drug_cols = [c for c in drug_cols if c in drug_metadata.columns]
        scores_df = scores_df.merge(
            drug_metadata[drug_cols],
            on='drug',
            how='left'
        )
    
    # merge cell line info
    if 'cell_line_id' in scores_df.columns:
        # cell line metadata may have multiple rows per cell line (one per driver mutation)
        # take first row per cell line for simplicity
        cell_cols = ['cell_name', 'Organ', 'Driver_Gene_Symbol']
        cell_cols = [c for c in cell_cols if c in cell_line_metadata.columns]
        cell_summary = cell_line_metadata.groupby('cell_name').first().reset_index()[cell_cols]
        
        scores_df = scores_df.merge(
            cell_summary,
            left_on='cell_line_id',
            right_on='cell_name',
            how='left'
        )
    
    return scores_df

In [14]:
# enrich each pathway's scores with metadata
if len(de_signatures) > 0 and pathway_scores_list:
    enriched_scores = []
    for scores in pathway_scores_list:
        enriched = enrich_with_metadata(scores, drug_metadata, cell_line_metadata)
        enriched_scores.append(enriched)
    
    # show example
    print("Enriched pathway scores (example):")
    print(enriched_scores[0].head())

Enriched pathway scores (example):
       drug  concentration  proliferation_log2fc  proliferation_n_genes  \
0    4EGI-1           0.05              0.039984                     22   
1  9-ING-41           0.05              0.157286                     22   

   proliferation_padj  proliferation_pct_change        pathway  \
0            0.128359                  2.810245  proliferation   
1            0.013264                 11.518719  proliferation   

                      moa-fine             moa-broad targets  
0  Protein synthesis inhibitor  inhibitor/antagonist   EIF4E  
1               GSK3 inhibitor  inhibitor/antagonist   GSK3B  


In [16]:
print(enriched_scores)

[       drug  concentration  proliferation_log2fc  proliferation_n_genes  \
0    4EGI-1           0.05              0.039984                     22   
1  9-ING-41           0.05              0.157286                     22   

   proliferation_padj  proliferation_pct_change        pathway  \
0            0.128359                  2.810245  proliferation   
1            0.013264                 11.518719  proliferation   

                      moa-fine             moa-broad targets  
0  Protein synthesis inhibitor  inhibitor/antagonist   EIF4E  
1               GSK3 inhibitor  inhibitor/antagonist   GSK3B  ,        drug  concentration  apoptosis_log2fc  apoptosis_n_genes  \
0    4EGI-1           0.05         -0.154550                 13   
1  9-ING-41           0.05         -0.051164                 13   

   apoptosis_padj  apoptosis_pct_change    pathway  \
0        0.243384            -10.158727  apoptosis   
1        0.210264             -3.484256  apoptosis   

                   

## Summary

This notebook demonstrated:

1. **Loading metadata** - Gene, drug, cell line, and sample metadata are small enough to load fully
2. **Streaming DE data** - The pseudobulk differential expression data can be streamed to avoid downloading 50GB
3. **Filtering to pathways** - By filtering to signature genes first, we reduce data volume by ~95%
4. **Computing pathway scores** - Mean log2FoldChange across pathway genes gives a pathway activity score
5. **Enriching with metadata** - Drug MOA and cell line driver mutations provide biological context

For production use, consider:
- Adding more pathways from MSigDB (Hallmarks, KEGG, Reactome)
- Computing additional statistics (median, variance, gene-wise significance)
- Increasing `max_rows` to process more of the dataset